# SAC Training Workflow — Independent Model-Free SAC for PV Tracking

This notebook demonstrates the complete SAC training pipeline using the exact same MDP settings as MBPO:
- **6D Compact State Space**: `[DNI/2000, DHI/2000, P/(2000·A·η), β/90°, (γ%360°)/180°-1, (H-t)/H]`
- **Same Action Space**: Continuous `[delta_tilt, delta_azimuth]`
- **Same Reward Function**: Power - movement penalty
- **Same Simulator**: `pvlib` solar geometry + NSRDB weather scenarios
- **Same Episode Horizon**: 117 steps (5-minute intervals, 12:00-21:45 UTC)

**Key Difference**: This is pure SAC (model-free) — no model rollouts, no surrogate dynamics, no MBPO pipeline dependencies.

## 1. Setup & Configuration — Identical MDP Settings

In [ ]:
# Import modules from our independent SAC workspace
import sys
sys.path.insert(0, '.')

import numpy as np
import torch
import logging

# SAC workspace modules (independent from MBPO algorithms)
from SAC_workspace.src.representation.representation import StateEncoder, StateNormalizer
from SAC_workspace.src.environment.environment import SACEnvironment
from SAC_workspace.src.agents.sac_agent import SACAgent
from SAC_workspace.src.training.trainer import SACTrainer
from SAC_workspace.src.evaluation.evaluation import SACEvaluator

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(message)s')
logger = logging.getLogger('SAC_Notebook')

# Configuration matching MBPO conf_sac / conf_compact
CONFIG = {
    'observation_mode': 'compact',  # 6D state space
    'state_dim': 6,
    'action_dim': 2,
    'hidden_dim': 256,
    'episode_steps': 117,  # Same horizon as MBPO: 585 min / 5 min = 117 transitions
    'gamma': 0.99,
    'lr': 3e-4,
    'batch_size': 256,
    'replay_size': 100000,
    'alpha_init': 0.2,
    'automatic_entropy_tuning': True,
}

print('SAC Workspace Setup Complete')
print(f"Configuration: {CONFIG}")
print(f"State Space: 6D Compact [DNI/2000, DHI/2000, P/(2000·A·η), β/90°, (γ/180-1), (H-t)/H]")
print(f"Action Space: 2D [delta_tilt, delta_azimuth]")
print(f"Episode Horizon: {CONFIG['episode_steps']} steps (5-min intervals)")

## 2. Environment & Replay Buffer — Same pvlib + NSRDB Simulator

In [ ]:
# Initialize SAC environment (same pvlib physics, same NSRDB scenarios)
env_config = {
    'observation_mode': 'compact',
    'latitude': 40.72,
    'longitude': -74.01,
    'tz': 'UTC',
    'start_time': '12:00',
    'periods': 118,  # 118 timestamps → 117 steps
    'freq': '5min',
    'efficiency': 0.18,
    'area': 1.0,
    'max_delta_tilt': 5.0,
    'max_delta_azimuth': 10.0,
    'movement_penalty': 0.0,  # Same as SAC baseline
}

env = SACEnvironment(**env_config)
print('Environment initialized with same pvlib simulator as MBPO')
print(f"State dimension: {env.get_observation_space_info()['state_dim']}")
print(f"Action dimension: {env.get_action_space_info()['action_dim']}")
print(f"Episode horizon: {env.get_episode_length()} steps")

# Initialize replay buffer (same capacity as MBPO replay setup)
replay_buffer_size = 100000  # Standard SAC replay size
print(f"Replay buffer capacity: {replay_buffer_size}")

## 3. Agent Training — Pure SAC (No Model Rollouts)

In [ ]:
# Initialize SAC agent with mathematically rigorous formulation
agent = SACAgent(
    state_dim=6,
    action_dim=2,
    hidden_dim=256,
    lr=3e-4,
    gamma=0.99,
    tau=0.005,  # Polyak averaging coefficient
    alpha_init=0.2,
    batch_size=256,
    replay_size=100000,
    automatic_entropy_tuning=True,
)
print('SAC Agent initialized with twin critics and automatic entropy tuning')
print(f"Agent components: Actor π_φ, Twin Critics Q_θ1/Q_θ2, Target Networks Q̄_θ1/Q̄_θ2, Entropy α")

In [ ]:
# Initialize SAC trainer (model-free: no rollouts, no surrogate dynamics)
trainer = SACTrainer(
    env_kwargs=env_config,
    agent_kwargs={
        'state_dim': 6,
        'action_dim': 2,
        'hidden_dim': 256,
        'lr': 3e-4,
        'gamma': 0.99,
        'tau': 0.005,
        'alpha_init': 0.2,
        'batch_size': 256,
        'replay_size': 100000,
        'automatic_entropy_tuning': True,
    },
    training_config={
        'n_epochs': 2000,  # Same as MBPO
        'n_initial_exploration_steps': 8000,
        'episode_length': 117,
        'eval_interval': 10,
        'save_interval': 50,
        'log_interval': 1,
    },
    output_dir='SAC_workspace/outputs',
)
print('SAC Trainer initialized — pure model-free training loop (no rollouts)')
print('Note: All dynamics come directly from pvlib + NSRDB simulator (same as MBPO)')

In [ ]:
# Execute SAC training (model-free — each episode uses only real simulator transitions)
n_epochs = 10  # Short demo; use 2000 for full comparison with MBPO
results = trainer.train(n_epochs=_n_epochs)

print(f"Training completed: {results['final_checkpoint_path']}")
print(f"Mean final 10-episode reward: {results['mean_final_rewards']:.2f}")

## 4. Evaluation & Benchmarking — SAC vs Physical Oracle

In [ ]:
# Load trained SAC agent
checkpoint_path = 'SAC_workspace/outputs/sac_final_checkpoint.pt'
evaluator = SACEvaluator(agent_checkpoint_path=checkpoint_path)

# Evaluate SAC policy (deterministic: no exploration noise)
sac_metrics = evaluator.evaluate_sac_policy(num_episodes=10)
print('SAC Policy Evaluation Results:')
for k, v in sac_metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# Compare SAC against theoretical oracle (pvlib sun tracking + POA irradiance)
oracle_metrics = evaluator.evaluate_physical_oracle(num_episodes=10)
comparison = evaluator.compare_sac_vs_baseline(sac_metrics, oracle_metrics)

print('\nSAC vs Oracle Comparison:')
print(f"  SAC Mean Reward:     {comparison['sac_mean_reward']:.2f}")
print(f"  Oracle Mean Reward: {comparison['baseline_mean_reward']:.2f}")
print(f"  Difference:          {comparison['reward_difference']:.2f}")
print(f"  Relative Performance: {comparison['relative_performance']:.2%}")
print(f"Note: Oracle uses pvlib solar_zenith + solar_azimuth + POA irradiance for perfect tracking")

---
**Summary**: This notebook demonstrates the complete independent SAC pipeline:
- Same 6D compact state space as MBPO
- Same pvlib + NSRDB simulator
- Same 117-step episode horizon
- Pure SAC (model-free) — no rollouts, no surrogate dynamics
- Direct comparison with pvlib-based oracle

This ensures a fair, mathematically rigorous comparison: SAC vs MBPO on identical MDP tasks.